In [ ]:
def plot_voxel_tensor_plotly(tensor, duration=6, fps=30):
    assert tensor.ndim == 3

    nx, ny, nz = tensor.shape
    filled = tensor.astype(bool)

    # === ВОКСЕЛИ ===
    x, y, z = np.where(filled)

    voxels = go.Scatter3d(
        x=x + 0.5,
        y=y + 0.5,
        z=z + 0.5,
        mode='markers',
        marker=dict(
            size=6,
            color='black',
            opacity=0.9
        ),
        name='voxels'
    )

    # === ПРОЕКЦИЯ НА XoY ===
    projection = np.any(filled, axis=2)

    px, py = np.where(projection)
    pz = np.zeros_like(px)

    projection_points = go.Scatter3d(
        x=px + 0.5,
        y=py + 0.5,
        z=pz,
        mode='markers',
        marker=dict(
            size=4,
            color='black',
            opacity=0.4
        ),
        name='projection'
    )

    # === СЕТКА НА ПЛОСКОСТИ ===
    grid_lines = []

    for i in range(nx + 1):
        grid_lines.append(go.Scatter3d(
            x=[i, i], y=[0, ny], z=[0, 0],
            mode='lines',
            line=dict(color='black', width=2),
            showlegend=False
        ))

    for j in range(ny + 1):
        grid_lines.append(go.Scatter3d(
            x=[0, nx], y=[j, j], z=[0, 0],
            mode='lines',
            line=dict(color='black', width=2),
            showlegend=False
        ))

    # === ОСИ ===
    axes = [
        go.Scatter3d(x=[0, nx], y=[0, 0], z=[0, 0],
                     mode='lines', line=dict(color='red', width=5), name='X'),
        go.Scatter3d(x=[0, 0], y=[0, ny], z=[0, 0],
                     mode='lines', line=dict(color='green', width=5), name='Y'),
        go.Scatter3d(x=[0, 0], y=[0, 0], z=[0, nz],
                     mode='lines', line=dict(color='blue', width=5), name='Z'),
    ]

    # === ВСЕ ТРЕЙСЫ ===
    data = [voxels, projection_points] + grid_lines + axes

    # === АНИМАЦИЯ ВРАЩЕНИЯ ===
    frames = []
    n_frames = int(duration * fps)

    for k, angle in enumerate(np.linspace(0, 2 * np.pi, n_frames)):
        camera = dict(
            eye=dict(
                x=2 * np.cos(angle),
                y=2 * np.sin(angle),
                z=1.5
            )
        )

        frames.append(go.Frame(
            layout=dict(scene_camera=camera)
        ))

    # === ФИГУРА ===
    fig = go.Figure(
        data=data,
        frames=frames
    )

    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data'
        ),
        margin=dict(l=0, r=0, t=40, b=0),
        updatemenus=[dict(
            type='buttons',
            showactive=False,
            buttons=[dict(
                label='Play',
                method='animate',
                args=[None, dict(
                    frame=dict(duration=1000 / fps, redraw=True),
                    fromcurrent=True
                )]
            )]
        )]
    )

    return fig

def GetMatrixFromFile(N: int):
    arr = []
    filename = f'f_tensor_file{N}.txt'
    with open(filename, 'r') as file:
        for line in file:
            # Пропускаем пустые строки
            line = line.strip()
            if not line:
                continue

            # Разделяем строку на части
            parts = line.split()
            if len(parts) >= 3:
                a, b, c = int(parts[0]), int(parts[1]), int(parts[2])
                arr.append((a, b, c))

    return arr

# ==== Пример использования ====
def main():
    # Пример: случайный тензор 5x5x5

    #tensor = np.random.randint(0, 2, size=(15, 15, 15))
    N = 5
    tensor = np.zeros((N * N - 1, N * N - 1, N * N - 1))
    arr = GetMatrixFromFile(N)
    for (one, two, three) in arr:
        tensor[one][two][three] = 1


    fig = plot_voxel_tensor_plotly(tensor)
    fig.show()

main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib

def plot_voxel_tensor_animated(tensor, rotation_angle=360, fps=30, duration=6):
    assert tensor.ndim == 3

    filled = tensor.astype(bool)

    # Цвета: можно задать для каждого вокселя
    colors = np.empty(filled.shape, dtype=object)
    colors[filled] = 'lightblue'        # закрашенные (1)


    fig = plt.figure(figsize=(10, 10))

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(projection='3d')



    # Находим проекцию на плоскость XoY
    projection = np.any(filled, axis=2)

    for x in range(tensor.shape[0]):
        for y in range(tensor.shape[1]):
            if projection[x, y]:
                # Создаем координаты для одного квадрата
                X = np.array([[x, x+1], [x, x+1]])
                Y = np.array([[y, y], [y+1, y+1]])
                Z = np.zeros((2, 2))

                # Рисуем один закрашенный квадрат
                ax.plot_surface(X, Y, Z,
                              color='lightgray',
                              alpha=1,
                              edgecolor='black',
                              linewidth=0.5)

    # Отключаем стандартные оси
    #ax.grid(False)
    #ax.axis('off')


    #ax.set_xlim(0, tensor.shape[0])
    #ax.set_ylim(0, tensor.shape[0])
    ax.set_zlim(0, tensor.shape[0])
    ax.set_aspect('equal')
    ax.set_xticks(range(tensor.shape[0] + 1))
    ax.set_yticks(range(tensor.shape[0] + 1))
    ax.set_zticks(range(tensor.shape[0] + 1))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_zticklabels([])

    ax.xaxis.line.set_color('white')
    ax.yaxis.line.set_color('white')
    ax.zaxis.line.set_color('white')

    # Скрыть всё, что связано с тиками
    ax.tick_params(axis='both', which='both',
               length=0,           # длина рисок = 0
               labelsize=0,        # размер шрифта = 0
               labelcolor='white', # цвет подписей = белый (если фон белый)
               colors='white',     # цвет рисок = белый
               pad=0)              # отступ = 0

    # Рисуем линии сетки на плоскости XoY (z=0)
    x_range = np.arange(tensor.shape[0] + 1)
    y_range = np.arange(tensor.shape[1] + 1)

    for x in x_range:
        ax.plot([x, x], [0, tensor.shape[1]], [0, 0],
                color='black', linewidth=1, alpha=1)
    for y in y_range:
        ax.plot([0, tensor.shape[0]], [y, y], [0, 0],
                color='black', linewidth=1, alpha=1)


    # Рисуем свои оси из точки (0,0,0)
    # Ось X (красная)
    ax.plot([0, tensor.shape[0]], [0, 0], [0, 0],
            color='red', linewidth=2, label='X')
    # Ось Y (зеленая)
    ax.plot([0, 0], [0, tensor.shape[1]], [0, 0],
            color='green', linewidth=2, label='Y')
    # Ось Z (синяя)
    ax.plot([0, 0], [0, 0], [0, tensor.shape[2]],
            color='blue', linewidth=2, label='Z')

    # рисуем линии, которые формируют очертания куба
    ax.plot([tensor.shape[0], tensor.shape[0]], [tensor.shape[0], tensor.shape[0]], [0, tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([tensor.shape[0], tensor.shape[0]], [0, tensor.shape[0]], [tensor.shape[0], tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([0, tensor.shape[0]], [tensor.shape[0], tensor.shape[0]], [tensor.shape[0], tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([0, tensor.shape[0]], [0, 0], [tensor.shape[0], tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([tensor.shape[0], tensor.shape[0]], [0, 0], [0, tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([0, 0], [tensor.shape[0], tensor.shape[0]], [0, tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([0, 0], [0, tensor.shape[0]], [tensor.shape[0], tensor.shape[0]],
            color='black', linewidth=1)


    # Добавляем стрелки на концах осей
    arrow_length = max(tensor.shape) * 0.05

    # Стрелка для X
    ax.quiver(tensor.shape[0], 0, 0, arrow_length, 0, 0,
              color='red', arrow_length_ratio=0.3)
    # Стрелка для Y
    ax.quiver(0, tensor.shape[1], 0, 0, arrow_length, 0,
              color='green', arrow_length_ratio=0.3)
    # Стрелка для Z
    ax.quiver(0, 0, tensor.shape[2], 0, 0, arrow_length,
              color='blue', arrow_length_ratio=0.3)


    # Добавляем подписи осей в концах
    ax.text(tensor.shape[0] + arrow_length, 0, 0, 'X', color='red', fontsize=12)
    ax.text(0, tensor.shape[1] + arrow_length, 0, 'Y', color='green', fontsize=12)
    ax.text(0, 0, tensor.shape[2] + arrow_length, 'Z', color='blue', fontsize=12)

    ax.voxels(filled, facecolors=colors, edgecolor='black', shade=False)


    # Функция обновления для анимации
    def update(frame):
        ax.view_init(elev=30, azim=frame)
        ax.set_title(f'Angle: {frame:.0f}°')
        return ax,

    # Создаем анимацию
    frames = np.linspace(0, rotation_angle, int(fps * duration))
    anim = FuncAnimation(fig, update, frames=frames, interval=1000 / fps, blit=False)

    # Для Google Colab используем HTML
    return HTML(anim.to_jshtml())


def GetMatrixFromFile(N: int):
    arr = []
    filename = f'f_tensor_file{N}.txt'
    with open(filename, 'r') as file:
        for line in file:
            # Пропускаем пустые строки
            line = line.strip()
            if not line:
                continue

            # Разделяем строку на части
            parts = line.split()
            if len(parts) >= 2:
                a, b, c = int(parts[0]), int(parts[1]), int(parts[2])
                arr.append((a, b, c))

    return arr

# ==== Пример использования ====
def main():
    N = 5
    tensor = np.zeros((N * N - 1, N * N - 1, N * N - 1))
    arr = GetMatrixFromFile(N)
    for (one, two, three) in arr:
        tensor[one][two][three] = 1


    matplotlib.rcParams['animation.embed_limit'] = 140
    anim = plot_voxel_tensor_animated(tensor, rotation_angle=360, fps=30, duration=8)
    display(anim)  # Явно отображаем анимацию

main()

In [ ]:
!apt-get install ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib

def plot_voxel_tensor_animated(tensor, rotation_angle=360, fps=30, duration=6):
    assert tensor.ndim == 3

    filled = tensor.astype(bool)

    # Цвета: можно задать для каждого вокселя
    colors = np.empty(filled.shape, dtype=object)
    colors[filled] = 'lightblue'        # закрашенные (1)


    fig = plt.figure(figsize=(11, 11))
    ax = fig.add_subplot(projection='3d')



    # Находим проекцию на плоскость XoY
    projection = np.any(filled, axis=2)

    for x in range(tensor.shape[0]):
        for y in range(tensor.shape[1]):
            if projection[x, y]:
                # Создаем координаты для одного квадрата
                X = np.array([[x, x+1], [x, x+1]])
                Y = np.array([[y, y], [y+1, y+1]])
                Z = np.zeros((2, 2))

                # Рисуем один закрашенный квадрат
                ax.plot_surface(X, Y, Z,
                              color='lightgray',
                              alpha=1,
                              edgecolor='black',
                              linewidth=0.5)

    # Отключаем стандартные оси
    #ax.grid(False)
    #ax.axis('off')


    #ax.set_xlim(0, tensor.shape[0])
    #ax.set_ylim(0, tensor.shape[0])
    ax.set_zlim(0, tensor.shape[0])
    ax.set_aspect('equal')
    ax.set_xticks(range(tensor.shape[0] + 1))
    ax.set_yticks(range(tensor.shape[0] + 1))
    ax.set_zticks(range(tensor.shape[0] + 1))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_zticklabels([])

    ax.xaxis.line.set_color('white')
    ax.yaxis.line.set_color('white')
    ax.zaxis.line.set_color('white')

    # Скрыть всё, что связано с тиками
    ax.tick_params(axis='both', which='both',
               length=0,           # длина рисок = 0
               labelsize=0,        # размер шрифта = 0
               labelcolor='white', # цвет подписей = белый (если фон белый)
               colors='white',     # цвет рисок = белый
               pad=0)              # отступ = 0

    # Рисуем линии сетки на плоскости XoY (z=0)
    x_range = np.arange(tensor.shape[0] + 1)
    y_range = np.arange(tensor.shape[1] + 1)

    for x in x_range:
        ax.plot([x, x], [0, tensor.shape[1]], [0, 0],
                color='black', linewidth=1, alpha=1)
    for y in y_range:
        ax.plot([0, tensor.shape[0]], [y, y], [0, 0],
                color='black', linewidth=1, alpha=1)


    # Рисуем свои оси из точки (0,0,0)
    # Ось X (красная)
    ax.plot([0, tensor.shape[0]], [0, 0], [0, 0],
            color='red', linewidth=2, label='X')
    # Ось Y (зеленая)
    ax.plot([0, 0], [0, tensor.shape[1]], [0, 0],
            color='green', linewidth=2, label='Y')
    # Ось Z (синяя)
    ax.plot([0, 0], [0, 0], [0, tensor.shape[2]],
            color='blue', linewidth=2, label='Z')

    # рисуем линии, которые формируют очертания куба
    ax.plot([tensor.shape[0], tensor.shape[0]], [tensor.shape[0], tensor.shape[0]], [0, tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([tensor.shape[0], tensor.shape[0]], [0, tensor.shape[0]], [tensor.shape[0], tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([0, tensor.shape[0]], [tensor.shape[0], tensor.shape[0]], [tensor.shape[0], tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([0, tensor.shape[0]], [0, 0], [tensor.shape[0], tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([tensor.shape[0], tensor.shape[0]], [0, 0], [0, tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([0, 0], [tensor.shape[0], tensor.shape[0]], [0, tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([0, 0], [0, tensor.shape[0]], [tensor.shape[0], tensor.shape[0]],
            color='black', linewidth=1)


    # Добавляем стрелки на концах осей
    arrow_length = max(tensor.shape) * 0.05

    # Стрелка для X
    ax.quiver(tensor.shape[0], 0, 0, arrow_length, 0, 0,
              color='red', arrow_length_ratio=0.3)
    # Стрелка для Y
    ax.quiver(0, tensor.shape[1], 0, 0, arrow_length, 0,
              color='green', arrow_length_ratio=0.3)
    # Стрелка для Z
    ax.quiver(0, 0, tensor.shape[2], 0, 0, arrow_length,
              color='blue', arrow_length_ratio=0.3)


    # Добавляем подписи осей в концах
    ax.text(tensor.shape[0] + arrow_length, 0, 0, 'X', color='red', fontsize=12)
    ax.text(0, tensor.shape[1] + arrow_length, 0, 'Y', color='green', fontsize=12)
    ax.text(0, 0, tensor.shape[2] + arrow_length, 'Z', color='blue', fontsize=12)

    ax.voxels(filled, facecolors=colors, edgecolor='black', shade=False)


    # Функция обновления для анимации
    def update(frame):
        ax.view_init(elev=30, azim=frame)
        ax.set_title(f'Angle: {frame:.0f}°')
        return ax,

    # Создаем анимацию
    frames = np.linspace(0, rotation_angle, int(fps * duration))
    anim = FuncAnimation(fig, update, frames=frames, interval=1000 / fps, blit=False)

    anim.save('animation.gif', writer='pillow', fps=fps)
    # Для Google Colab используем HTML
    return HTML(anim.to_jshtml())


def GetMatrixFromFile(N: int):
    arr = []
    filename = f'd_tensor_file{N}.txt'
    with open(filename, 'r') as file:
        for line in file:
            # Пропускаем пустые строки
            line = line.strip()
            if not line:
                continue

            # Разделяем строку на части
            parts = line.split()
            if len(parts) >= 2:
                a, b, c = int(parts[0]), int(parts[1]), int(parts[2])
                arr.append((a, b, c))

    return arr

# ==== Пример использования ====
def main():
    N = 5
    tensor = np.zeros((N * N - 1, N * N - 1, N * N - 1))
    arr = GetMatrixFromFile(N)
    for (one, two, three) in arr:
        tensor[one][two][three] = 1


    matplotlib.rcParams['animation.embed_limit'] = 300
    anim = plot_voxel_tensor_animated(tensor, rotation_angle=360, fps=30, duration=10)
    display(anim)  # Явно отображаем анимацию

main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib

def plot_voxel_tensor_animated(tensor, rotation_angle=360, fps=30, duration=6):
    assert tensor.ndim == 3

    filled = tensor.astype(bool)

    # Цвета: можно задать для каждого вокселя
    colors = np.empty(filled.shape, dtype=object)
    colors[filled] = 'lightblue'        # закрашенные (1)


    fig = plt.figure(figsize=(11, 11))
    ax = fig.add_subplot(projection='3d')



    # Находим проекцию на плоскость XoY
    projection = np.any(filled, axis=2)

    for x in range(tensor.shape[0]):
        for y in range(tensor.shape[1]):
            if projection[x, y]:
                # Создаем координаты для одного квадрата
                X = np.array([[x, x+1], [x, x+1]])
                Y = np.array([[y, y], [y+1, y+1]])
                Z = np.zeros((2, 2))

                # Рисуем один закрашенный квадрат
                ax.plot_surface(X, Y, Z,
                              color='lightgray',
                              alpha=1,
                              edgecolor='black',
                              linewidth=0.5)

    # Отключаем стандартные оси
    #ax.grid(False)
    #ax.axis('off')


    #ax.set_xlim(0, tensor.shape[0])
    #ax.set_ylim(0, tensor.shape[0])
    ax.set_zlim(0, tensor.shape[0])
    ax.set_aspect('equal')
    ax.set_xticks(range(tensor.shape[0] + 1))
    ax.set_yticks(range(tensor.shape[0] + 1))
    ax.set_zticks(range(tensor.shape[0] + 1))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_zticklabels([])

    ax.xaxis.line.set_color('white')
    ax.yaxis.line.set_color('white')
    ax.zaxis.line.set_color('white')

    # Скрыть всё, что связано с тиками
    ax.tick_params(axis='both', which='both',
               length=0,           # длина рисок = 0
               labelsize=0,        # размер шрифта = 0
               labelcolor='white', # цвет подписей = белый (если фон белый)
               colors='white',     # цвет рисок = белый
               pad=0)              # отступ = 0

    # Рисуем линии сетки на плоскости XoY (z=0)
    x_range = np.arange(tensor.shape[0] + 1)
    y_range = np.arange(tensor.shape[1] + 1)

    for x in x_range:
        ax.plot([x, x], [0, tensor.shape[1]], [0, 0],
                color='black', linewidth=1, alpha=1)
    for y in y_range:
        ax.plot([0, tensor.shape[0]], [y, y], [0, 0],
                color='black', linewidth=1, alpha=1)


    # Рисуем свои оси из точки (0,0,0)
    # Ось X (красная)
    ax.plot([0, tensor.shape[0]], [0, 0], [0, 0],
            color='red', linewidth=2, label='X')
    # Ось Y (зеленая)
    ax.plot([0, 0], [0, tensor.shape[1]], [0, 0],
            color='green', linewidth=2, label='Y')
    # Ось Z (синяя)
    ax.plot([0, 0], [0, 0], [0, tensor.shape[2]],
            color='blue', linewidth=2, label='Z')

    # рисуем линии, которые формируют очертания куба
    ax.plot([tensor.shape[0], tensor.shape[0]], [tensor.shape[0], tensor.shape[0]], [0, tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([tensor.shape[0], tensor.shape[0]], [0, tensor.shape[0]], [tensor.shape[0], tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([0, tensor.shape[0]], [tensor.shape[0], tensor.shape[0]], [tensor.shape[0], tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([0, tensor.shape[0]], [0, 0], [tensor.shape[0], tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([tensor.shape[0], tensor.shape[0]], [0, 0], [0, tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([0, 0], [tensor.shape[0], tensor.shape[0]], [0, tensor.shape[0]],
            color='black', linewidth=1)

    ax.plot([0, 0], [0, tensor.shape[0]], [tensor.shape[0], tensor.shape[0]],
            color='black', linewidth=1)


    # Добавляем стрелки на концах осей
    arrow_length = max(tensor.shape) * 0.05

    # Стрелка для X
    ax.quiver(tensor.shape[0], 0, 0, arrow_length, 0, 0,
              color='red', arrow_length_ratio=0.3)
    # Стрелка для Y
    ax.quiver(0, tensor.shape[1], 0, 0, arrow_length, 0,
              color='green', arrow_length_ratio=0.3)
    # Стрелка для Z
    ax.quiver(0, 0, tensor.shape[2], 0, 0, arrow_length,
              color='blue', arrow_length_ratio=0.3)


    # Добавляем подписи осей в концах
    ax.text(tensor.shape[0] + arrow_length, 0, 0, 'X', color='red', fontsize=12)
    ax.text(0, tensor.shape[1] + arrow_length, 0, 'Y', color='green', fontsize=12)
    ax.text(0, 0, tensor.shape[2] + arrow_length, 'Z', color='blue', fontsize=12)

    ax.voxels(filled, facecolors=colors, edgecolor='black', shade=False)


    # Функция обновления для анимации
    def update(frame):
        ax.view_init(elev=30, azim=frame)
        ax.set_title(f'Angle: {frame:.0f}°')
        return ax,

    # Создаем анимацию
    frames = np.linspace(0, rotation_angle, int(fps * duration))
    anim = FuncAnimation(fig, update, frames=frames, interval=1000 / fps, blit=False)

    anim.save('animation.gif', writer='pillow', fps=fps)
    # Для Google Colab используем HTML
    return HTML(anim.to_jshtml())


def GetMatrixFromFile(N: int):
    arr = []
    filename = f'f_tensor_file{N}.txt'
    with open(filename, 'r') as file:
        for line in file:
            # Пропускаем пустые строки
            line = line.strip()
            if not line:
                continue

            # Разделяем строку на части
            parts = line.split()
            if len(parts) >= 2:
                a, b, c = int(parts[0]), int(parts[1]), int(parts[2])
                arr.append((a, b, c))

    return arr

# ==== Пример использования ====
def main():
    N = 6
    tensor = np.zeros((N * N - 1, N * N - 1, N * N - 1))
    arr = GetMatrixFromFile(N)
    for (one, two, three) in arr:
        tensor[one][two][three] = 1


    matplotlib.rcParams['animation.embed_limit'] = 300
    anim = plot_voxel_tensor_animated(tensor, rotation_angle=360, fps=30, duration=10)
    display(anim)  # Явно отображаем анимацию

main()